
# Diagnóstico de inadimplência por fotografia — safra × idade × época

Versão consolidada, pronta para receber a base bruta real. Ponto de
entrada único: um `DataFrame` Spark chamado `df_spark_bruto` com estas
colunas (os nomes exatos que você usa):

| Coluna | O que é |
|---|---|
| `id_contrato` | identificador único do contrato |
| `id_cliente` | identificador único do cliente (**não usado neste notebook** — ver nota no final) |
| `data_originacao` | data de nascimento do contrato (define a safra) |
| `data_ref` | data da fotografia daquela linha |
| `flag_default` | **estado** (1 = em atraso ≥90 dias naquele mês; 0 = fora) -- pode curar e voltar a 0, e depois entrar de novo |

A Seção 1 gera uma base sintética **só para o notebook rodar de ponta a
ponta e você validar a lógica antes de trocar pela real** -- é o único
bloco a apagar e substituir por `df_spark_bruto = spark.read...` (ou
equivalente) quando for usar dado de verdade. Tudo dali para frente
funciona só com o schema da tabela acima, sem depender de nada específico
do gerador sintético.


In [1]:

import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

pd.set_option("display.max_rows", 30)
spark = SparkSession.builder.appName("carteira_diagnostico").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 15:37:55 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/28 15:37:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/28 15:37:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



## Configuração

Nenhum destes é lido do dado -- são escolhas que você faz e pode ajustar
depois de ver os primeiros resultados.


In [2]:

USA_MACRO = True            # False = fica sem ancora de epoca (ver Secao 4)
LARGURA_MOB_BIN = 3         # meses por faixa de idade
LIMIAR_CREDIVEL = 6         # meses observados minimos para confiar no efeito de safra (Secao 8)
K_REFERENCIA = 0.5          # folga do teste de Page (Secao 6)
PERCENTIL_H = 95            # percentil usado para calibrar o limite de alarme
N_REPLICACOES_NULAS = 300   # repeticoes da simulacao nula para calibrar h
POSICAO_INICIO_BACKTEST = 10  # indice (nao mes) a partir do qual o backtest comeca



## 1. Ponto de entrada -- SUBSTITUIR este bloco pela leitura real

Gera uma base sintética com exatamente o schema esperado (incluindo
estoque nascido antes da janela de observação, para testar o truncamento,
e cura/reentrada em atraso, para testar a elegibilidade da Seção 2).
Injeto de propósito uma leva de safras ruins e um choque de época
defasado -- resposta conhecida, para conferir se o pipeline recupera o
que foi injetado antes de confiar nele com dado real.

**Apague esta célula e troque por algo como**
`df_spark_bruto = spark.read.table("sua_tabela_de_atraso")`
**quando for rodar com a base real.**


In [3]:

def _gerar_base_sintetica_para_teste():
    rng = np.random.default_rng(42)
    PEAK_MOB, BASE_RATE, PROB_CURA = 7, 0.018, 0.15
    SAFRAS_RUINS_OFFSET = {9, 10, 11}
    EFEITO_COHORT_RUIM = 0.5
    INICIO_CHOQUE_MACRO_OFFSET = 15
    N_MESES_JANELA, N_MESES_PRE_JANELA, N_POR_SAFRA = 24, 12, 500
    DATA_INICIO_JANELA = pd.Timestamp("2022-01-01")

    def hazard_base(mob):
        return BASE_RATE * (mob / PEAK_MOB) * np.exp(1 - mob / PEAK_MOB)

    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    macro_verdadeiro = np.zeros(N_MESES_JANELA + 1)
    for t in range(1, N_MESES_JANELA + 1):
        if t >= INICIO_CHOQUE_MACRO_OFFSET:
            macro_verdadeiro[t] = min(0.9, (t - (INICIO_CHOQUE_MACRO_OFFSET - 1)) * 0.9 / 10)

    offset_min = 1 - N_MESES_PRE_JANELA
    linhas = []
    estado_default, contrato_ids, cliente_de_contrato = {}, {}, {}
    for s in range(offset_min, N_MESES_JANELA + 1):
        estado_default[s] = np.zeros(N_POR_SAFRA, dtype=bool)
        contrato_ids[s] = [f"CT{s:04d}_{i:05d}" for i in range(N_POR_SAFRA)]
        for i in range(N_POR_SAFRA):
            cliente_de_contrato[contrato_ids[s][i]] = f"CL{(s*7919 + i) % 300000:06d}"

    for t in range(offset_min, N_MESES_JANELA + 1):
        for s in range(offset_min, min(t, N_MESES_JANELA) + 1):
            mob = t - s + 1
            if mob < 1:
                continue
            em_default = estado_default[s]
            cohort_efeito = EFEITO_COHORT_RUIM if s in SAFRAS_RUINS_OFFSET else 0.0
            macro_t = macro_verdadeiro[t] if 1 <= t <= N_MESES_JANELA else 0.0
            h_base = hazard_base(mob)
            p_entrada = sigmoid(np.log(h_base / (1 - h_base)) + cohort_efeito + macro_t)

            novo_estado = em_default.copy()
            idx_default = np.where(em_default)[0]
            cura = rng.random(len(idx_default)) < PROB_CURA
            novo_estado[idx_default[cura]] = False
            idx_livre = np.where(~em_default)[0]
            entra = rng.random(len(idx_livre)) < p_entrada
            novo_estado[idx_livre[entra]] = True

            if t >= 1:
                data_ref = DATA_INICIO_JANELA + pd.DateOffset(months=t - 1)
                data_orig = DATA_INICIO_JANELA + pd.DateOffset(months=s - 1)
                ids = contrato_ids[s]
                linhas.append(pd.DataFrame({
                    "id_contrato": ids,
                    "id_cliente": [cliente_de_contrato[c] for c in ids],
                    "data_originacao": data_orig, "data_ref": data_ref,
                    "flag_default": novo_estado.astype(int),
                }))
            estado_default[s] = novo_estado

    return pd.concat(linhas, ignore_index=True)

df_spark_bruto = spark.createDataFrame(_gerar_base_sintetica_para_teste())
print(f"df_spark_bruto: {df_spark_bruto.count()} linhas")
df_spark_bruto.show(5)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; 

df_spark_bruto: 294000 linhas


+------------+----------+-------------------+-------------------+------------+
| id_contrato|id_cliente|    data_originacao|           data_ref|flag_default|
+------------+----------+-------------------+-------------------+------------+
|CT-011_00000|  CL212891|2021-01-01 00:00:00|2022-01-01 00:00:00|           0|
|CT-011_00001|  CL212892|2021-01-01 00:00:00|2022-01-01 00:00:00|           0|
|CT-011_00002|  CL212893|2021-01-01 00:00:00|2022-01-01 00:00:00|           0|
|CT-011_00003|  CL212894|2021-01-01 00:00:00|2022-01-01 00:00:00|           0|
|CT-011_00004|  CL212895|2021-01-01 00:00:00|2022-01-01 00:00:00|           0|
+------------+----------+-------------------+-------------------+------------+
only showing top 5 rows


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe



## 2. Derivar safra/MOB e elegibilidade (Spark)

Tudo em Spark porque é a etapa cara (grão contrato-mês). Três coisas
acontecem aqui, cada uma resolvendo um problema específico:

1. **`safra` e `mes_calendario`** truncados para o mês (`F.trunc`),
   robusto mesmo se a data vier com dia variável.
2. **`mob`** via diferença de meses entre as duas datas.
3. **Elegibilidade** -- como `flag_default` é estado (não evento) e pode
   curar e reentrar, "entrou este mês" é a transição 0→1, não o valor
   bruto da flag. Um contrato só entra na população em risco de um mês se
   sabemos com certeza que ele **não estava em default entrando naquele
   mês** -- ou porque é `mob==1` (nasceu ali, começo limpo garantido), ou
   porque no mês anterior a flag dele era 0. Quem tem `mob>1` e não tem
   mês anterior na base (veio truncado, com uma flag que já podia estar
   ligada há tempos) fica de fora -- não dá para saber se "entrou agora"
   ou "já estava assim antes da janela existir".


In [4]:

df = (
    df_spark_bruto
    .withColumn("safra", F.trunc("data_originacao", "month"))
    .withColumn("mes_calendario", F.trunc("data_ref", "month"))
    .withColumn("mob", (F.months_between("mes_calendario", "safra")).cast("int") + 1)
)

janela_contrato = Window.partitionBy("id_contrato").orderBy("data_ref")
df = df.withColumn("flag_anterior", F.lag("flag_default", 1).over(janela_contrato))

df_elegivel = df.filter((F.col("mob") == 1) | (F.col("flag_anterior") == 0))

checagem = df_elegivel.filter(F.col("flag_anterior") == 1).count()
print(f"Elegiveis que na verdade estavam em default no mes anterior (esperado 0): {checagem}")
assert checagem == 0, "Filtro de elegibilidade com furo -- nao seguir sem investigar"


Elegiveis que na verdade estavam em default no mes anterior (esperado 0): 0


## 3. Agregação Spark → pandas, com checagem cruzada

In [5]:

df_agregado = (
    df_elegivel
    .groupBy("safra", "mob", "mes_calendario")
    .agg(F.count("*").alias("populacao_risco"), F.sum("flag_default").alias("entrantes_90mais"))
)

t0 = time.time()
painel = df_agregado.toPandas()
painel["safra"] = pd.to_datetime(painel["safra"])
painel["mes_calendario"] = pd.to_datetime(painel["mes_calendario"])
print(f"Agregado Spark -> pandas em {time.time()-t0:.1f}s -- {len(painel)} celulas")

total_spark = int(painel["entrantes_90mais"].sum())
total_elegivel = df_elegivel.agg(F.sum("flag_default")).first()[0]
assert total_spark == total_elegivel, "Divergencia entre agregado e base elegivel -- nao seguir sem investigar"
print(f"Checagem cruzada ok: {total_spark} entrantes em ambos os calculos")


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Agregado Spark -> pandas em 4.5s -- 576 celulas


Checagem cruzada ok: 4271 entrantes em ambos os calculos



## 4. Derivações em pandas: truncamento, faixa de idade, macro

`veio_truncado` compara a safra contra o **início real da janela**, lido
do próprio dado bruto (menor `data_ref` que existe em toda a base) -- não
um valor fixado à mão.

Se `USA_MACRO=True`, este bloco espera um `df_macro` externo com colunas
`mes_calendario` + `macro_reportado` (mesclado por mês, aplicado a toda
safra viva naquele mês -- não por safra). **Sem essa base ainda, deixe
`USA_MACRO=False`** -- não crie uma variável fake para preencher a
lacuna; a Seção 5 explica a consequência de rodar sem ela.


In [6]:

INICIO_JANELA = df_spark_bruto.agg(F.min("data_ref")).first()[0]
INICIO_JANELA = pd.Timestamp(INICIO_JANELA).replace(day=1)
painel["veio_truncado"] = (painel["safra"] < INICIO_JANELA).astype(int)

mob_maximo = int(painel["mob"].max())
bins_mob = list(range(0, mob_maximo + LARGURA_MOB_BIN, LARGURA_MOB_BIN))
labels_mob = [f"{i+1}-{i+LARGURA_MOB_BIN}" for i in bins_mob[:-1]]
painel["mob_bin"] = pd.cut(painel["mob"], bins=bins_mob, labels=labels_mob)

painel["taxa_entrada"] = painel["entrantes_90mais"] / painel["populacao_risco"]

if USA_MACRO:
    # --- SUBSTITUIR pelo merge com sua base macro real (BCB, ou o fator
    #     prospectivo do overlay 4.966 se for reusar) ---
    _meses = sorted(painel["mes_calendario"].unique())
    df_macro = pd.DataFrame({
        "mes_calendario": _meses,
        "macro_reportado": np.random.default_rng(1).normal(0, 0.05, len(_meses)),
    })
    painel = painel.merge(df_macro, on="mes_calendario", how="left")

print(f"Inicio da janela detectado: {INICIO_JANELA.date()} | MOB maximo: {mob_maximo}")
painel.sort_values(["safra", "mob"]).head()


Inicio da janela detectado: 2022-01-01 | MOB maximo: 36


,safra,mob,mes_calendario,populacao_risco,entrantes_90mais,veio_truncado,mob_bin,taxa_entrada,macro_reportado
175,2021-01-01,14,2022-02-01,457,5,1,13-15,0.010941,0.041081
440,2021-01-01,15,2022-03-01,459,4,1,13-15,0.008715,0.016522
333,2021-01-01,16,2022-04-01,461,3,1,16-18,0.006508,-0.065158
406,2021-01-01,17,2022-05-01,461,8,1,16-18,0.017354,0.045268
412,2021-01-01,18,2022-06-01,461,7,1,16-18,0.015184,0.022319


### Checagem rápida: a inadimplência agregada está mesmo subindo?

In [7]:

total_mensal = painel.groupby("mes_calendario")["entrantes_90mais"].sum().reset_index()

fig_check = go.Figure()
fig_check.add_trace(go.Scatter(
    x=total_mensal["mes_calendario"], y=total_mensal["entrantes_90mais"],
    mode="lines+markers", name="Novos entrantes em 90+ / mes", line=dict(color="#B23A48", width=2),
))
fig_check.update_layout(
    title="Total de novos entrantes em atraso grave por mes de referencia (fotografia)",
    xaxis_title="Mes de referencia", yaxis_title="Novos entrantes",
    template="plotly_white", width=1400, height=420,
)
fig_check.show()



## 5. Modelo de referência

`entrantes_90mais ~ idade (faixas) + veio_truncado + [macro]`, offset =
log(população em risco). Sem `macro`, o efeito de época fica sem âncora
-- qualquer variação de mês cai inteira no resíduo da Seção 6, o teste
ainda funciona mas deixa de distinguir choque explicável de anomalia real.


In [8]:

def formula_modelo_a():
    base = "entrantes_90mais ~ C(mob_bin) + veio_truncado"
    return base + " + macro_reportado" if USA_MACRO else base

FORMULA_MODELO_A = formula_modelo_a()
print("Formula em uso:", FORMULA_MODELO_A)

def ajustar_modelo_a(dados_treino):
    return smf.glm(
        formula=FORMULA_MODELO_A, data=dados_treino,
        family=sm.families.Poisson(), offset=np.log(dados_treino["populacao_risco"]),
    ).fit()

modelo_referencia_completo = ajustar_modelo_a(painel)
print(modelo_referencia_completo.summary())


Formula em uso: entrantes_90mais ~ C(mob_bin) + veio_truncado + macro_reportado
                 Generalized Linear Model Regression Results                  
Dep. Variable:       entrantes_90mais   No. Observations:                  576
Model:                            GLM   Df Residuals:                      562
Model Family:                 Poisson   Df Model:                           13
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1498.9
Date:                Fri, 28 Aug 2026   Deviance:                       893.41
Time:                        15:38:32   Pearson chi2:                     892.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.7390
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------


## 6. Backtest um passo à frente

Ajusta só com meses anteriores, prevê o mês novo, acumula o resíduo
padronizado. `POSICAO_INICIO_BACKTEST` é um índice de posição na lista de
meses distintos (não um mês específico) -- se o primeiro ajuste der erro
de matriz singular, aumente esse número até sobrar variação suficiente em
cada faixa de MOB e (se `USA_MACRO`) mais de um mês de calendário no
treino.


In [9]:

meses_unicos = sorted(painel["mes_calendario"].unique())
resultados_backtest = []
for i in range(POSICAO_INICIO_BACKTEST, len(meses_unicos)):
    mes_atual = meses_unicos[i]
    treino = painel[painel["mes_calendario"] < mes_atual].copy()
    novo = painel[painel["mes_calendario"] == mes_atual].copy()
    if novo.empty:
        continue
    try:
        modelo_t = ajustar_modelo_a(treino)
        esperado = modelo_t.predict(novo, offset=np.log(novo["populacao_risco"]))
    except Exception as erro:
        print(f"Fotografia {mes_atual}: ajuste falhou ({erro}) -- pulando")
        continue

    n_celulas = len(novo)
    residuo_pearson = (
        (novo["entrantes_90mais"].values - esperado.values) / np.sqrt(np.maximum(esperado.values, 0.5))
    ).sum()
    resultados_backtest.append(dict(
        mes_calendario=mes_atual, observado=novo["entrantes_90mais"].sum(),
        esperado=esperado.sum(), residuo_pearson=residuo_pearson, n_celulas=n_celulas,
    ))

backtest_df = pd.DataFrame(resultados_backtest)
backtest_df["z"] = backtest_df["residuo_pearson"] / np.sqrt(backtest_df["n_celulas"])
backtest_df


,mes_calendario,observado,esperado,residuo_pearson,n_celulas,z
0,2022-11-01,172,143.944925,11.107428,23,2.316059
1,2022-12-01,150,150.896559,0.161769,24,0.033021
2,2023-01-01,142,155.743379,-6.087084,25,-1.217417
3,2023-02-01,142,149.569346,-0.985744,26,-0.193320
4,2023-03-01,156,152.299182,1.066273,27,0.205204
5,2023-04-01,185,163.500779,6.617327,28,1.250557
6,2023-05-01,193,162.103797,12.557690,29,2.331904
7,2023-06-01,222,167.963807,20.826090,30,3.802307
8,2023-07-01,269,179.613416,35.982916,31,6.462722
9,2023-08-01,272,191.955330,31.369998,32,5.545485



## 7. Teste de Page

$$C^+_t = \max(0,\; C^+_{t-1} + z_t - k) \qquad C^-_t = \max(0,\; C^-_{t-1} - z_t - k)$$

`k` (folga) impede que ruído comum se acumule; `h` (limite de decisão) é
calibrado simulando um cenário nulo (sem safra ruim, sem choque), **com a
escala do simulador extraída do seu painel real** (número de safras,
tamanho típico de coorte) em vez de constantes fixas -- isso não existia
nas versões anteriores e é a correção do ponto mais frágil delas: calibrar
contra um processo de escala parecida com a sua, não contra um número
arbitrário.

**Limite que continua valendo**: o simulador nulo abaixo não reproduz o
estoque herdado nem a cura/reentrada -- só a escala (nº de safras, nº de
meses, tamanho de coorte). Calibração aproximada, não exata.


In [10]:

def calcular_cusum_page(serie_z, k):
    c_mais, c_menos = np.zeros(len(serie_z)), np.zeros(len(serie_z))
    for i, z in enumerate(serie_z):
        ant_mais = c_mais[i - 1] if i > 0 else 0.0
        ant_menos = c_menos[i - 1] if i > 0 else 0.0
        c_mais[i] = max(0.0, ant_mais + z - k)
        c_menos[i] = max(0.0, ant_menos - z - k)
    return c_mais, c_menos

# escala extraida do painel real, nao fixada a mao
n_safras_sim = max(int(painel["mes_calendario"].nunique()), 6)
n_meses_sim = n_safras_sim
pop_mob1 = painel.loc[painel["mob"] == 1, "populacao_risco"]
n_por_safra_sim = int(pop_mob1.mean()) if len(pop_mob1) > 0 else int(
    painel.groupby("safra")["populacao_risco"].max().median()
)
print(f"Simulador nulo calibrado com: {n_safras_sim} safras, {n_por_safra_sim} contratos/safra")

def hazard_base_sim(mob, peak_mob=7, base_rate=0.018):
    return base_rate * (mob / peak_mob) * np.exp(1 - mob / peak_mob)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def simular_painel_nulo(rng):
    registros = []
    estado_ativo = {s: np.ones(n_por_safra_sim, dtype=bool) for s in range(1, n_meses_sim + 1)}
    for t in range(1, n_meses_sim + 1):
        for s in range(1, t + 1):
            mob = t - s + 1
            ativos = estado_ativo[s]
            n = int(ativos.sum())
            if n == 0:
                continue
            h_base = hazard_base_sim(mob)
            p = sigmoid(np.log(h_base / (1 - h_base)))
            sorteio = rng.random(n) < p
            idx = np.where(ativos)[0]
            estado_ativo[s][idx[sorteio]] = False
            registros.append(dict(mes_calendario=t, safra=s, mob=mob,
                                   entrantes_90mais=int(sorteio.sum()), populacao_risco=n))
    df_nulo = pd.DataFrame(registros)
    df_nulo["veio_truncado"] = 0
    bins_nulo = list(range(0, n_meses_sim + LARGURA_MOB_BIN, LARGURA_MOB_BIN))
    labels_nulo = [f"{i+1}-{i+LARGURA_MOB_BIN}" for i in bins_nulo[:-1]]
    df_nulo["mob_bin"] = pd.cut(df_nulo["mob"], bins=bins_nulo, labels=labels_nulo)
    if USA_MACRO:
        df_nulo["macro_reportado"] = 0.0
    return df_nulo

posicao_inicio_nulo = min(POSICAO_INICIO_BACKTEST, max(n_meses_sim - 3, 1))
maximos_c_mais_nulo = []
for i in range(N_REPLICACOES_NULAS):
    rng_i = np.random.default_rng(1000 + i)
    painel_nulo = simular_painel_nulo(rng_i)
    linhas = []
    for t_foto in range(posicao_inicio_nulo, n_meses_sim + 1):
        treino = painel_nulo[painel_nulo["mes_calendario"] < t_foto]
        novo = painel_nulo[painel_nulo["mes_calendario"] == t_foto]
        if novo.empty:
            continue
        try:
            modelo_i = ajustar_modelo_a(treino)
            esperado = modelo_i.predict(novo, offset=np.log(novo["populacao_risco"]))
        except Exception:
            continue
        n_celulas = len(novo)
        residuo = ((novo["entrantes_90mais"].values - esperado.values)
                   / np.sqrt(np.maximum(esperado.values, 0.5))).sum()
        linhas.append(residuo / np.sqrt(n_celulas))
    if len(linhas) < 3:
        continue
    c_mais_i, _ = calcular_cusum_page(np.array(linhas), K_REFERENCIA)
    maximos_c_mais_nulo.append(c_mais_i.max())

maximos_c_mais_nulo = np.array(maximos_c_mais_nulo)
H_EMPIRICO = float(np.percentile(maximos_c_mais_nulo, PERCENTIL_H))
print(f"Replicacoes nulas validas: {len(maximos_c_mais_nulo)} de {N_REPLICACOES_NULAS}")
print(f"h calibrado (percentil {PERCENTIL_H} do maximo sob H0): {H_EMPIRICO:.2f}")


Simulador nulo calibrado com: 24 safras, 500 contratos/safra


Replicacoes nulas validas: 300 de 300
h calibrado (percentil 95 do maximo sob H0): 5.97


In [11]:

c_mais, c_menos = calcular_cusum_page(backtest_df["z"].values, K_REFERENCIA)
backtest_df["c_mais"] = c_mais
backtest_df["c_menos"] = c_menos

meses_alarme_alta = backtest_df.loc[backtest_df["c_mais"] >= H_EMPIRICO, "mes_calendario"]
meses_alarme_queda = backtest_df.loc[backtest_df["c_menos"] >= H_EMPIRICO, "mes_calendario"]
print(f"Primeiro alarme de piora (C+ >= {H_EMPIRICO:.2f}): "
      f"{meses_alarme_alta.min().date() if not meses_alarme_alta.empty else 'nenhum'}")
print(f"Primeiro alarme de melhora (C- >= {H_EMPIRICO:.2f}): "
      f"{meses_alarme_queda.min().date() if not meses_alarme_queda.empty else 'nenhum'} "
      f"-- checar se e melhora real ou recalibracao do modelo antes de aceitar")

backtest_df[["mes_calendario", "z", "c_mais", "c_menos"]]


Primeiro alarme de piora (C+ >= 5.97): 2023-07-01
Primeiro alarme de melhora (C- >= 5.97): nenhum -- checar se e melhora real ou recalibracao do modelo antes de aceitar


,mes_calendario,z,c_mais,c_menos
0,2022-11-01,2.316059,1.816059,0.000000
1,2022-12-01,0.033021,1.349080,0.000000
2,2023-01-01,-1.217417,0.000000,0.717417
3,2023-02-01,-0.193320,0.000000,0.410737
4,2023-03-01,0.205204,0.000000,0.000000
5,2023-04-01,1.250557,0.750557,0.000000
6,2023-05-01,2.331904,2.582462,0.000000
7,2023-06-01,3.802307,5.884768,0.000000
8,2023-07-01,6.462722,11.847491,0.000000
9,2023-08-01,5.545485,16.892975,0.000000


### Visualização evoluindo fotografia por fotografia

In [12]:

safras_ordenadas = sorted(painel["safra"].unique())
mobs_ordenados = list(range(1, mob_maximo + 1))
mapa_linha_safra = {s: i for i, s in enumerate(safras_ordenadas)}

def matriz_revelada_ate(mes_foto):
    matriz = np.full((len(safras_ordenadas), len(mobs_ordenados)), np.nan)
    visivel = painel[painel["mes_calendario"] <= mes_foto]
    for _, linha in visivel.iterrows():
        i = mapa_linha_safra[linha["safra"]]
        j = int(linha["mob"]) - 1
        matriz[i, j] = linha["taxa_entrada"]
    return matriz

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Mapa safra x idade (MOB) -- taxa de entrada em 90+",
                     f"Teste de Page: C+ (piora) e C- (melhora), limite h={H_EMPIRICO:.1f}"),
    column_widths=[0.5, 0.5],
)

z0 = matriz_revelada_ate(meses_unicos[0])
fig.add_trace(go.Heatmap(
    z=z0, x=mobs_ordenados, y=[str(s.date()) for s in safras_ordenadas], colorscale="Reds",
    zmin=0, zmax=painel["taxa_entrada"].quantile(0.98), colorbar=dict(title="taxa", x=0.46),
), row=1, col=1)

fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="C+ (piora sustentada)",
                          line=dict(color="#B23A48", width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="C- (melhora sustentada)",
                          line=dict(color="#2A6F97", width=2)), row=1, col=2)
fig.add_hline(y=H_EMPIRICO, line_dash="dash", line_color="black",
              annotation_text=f"h = {H_EMPIRICO:.1f}", row=1, col=2)

frames = []
for mes_foto in meses_unicos:
    z_t = matriz_revelada_ate(mes_foto)
    sub = backtest_df[backtest_df["mes_calendario"] <= mes_foto]
    frames.append(go.Frame(
        name=str(mes_foto.date()),
        data=[
            go.Heatmap(z=z_t, x=mobs_ordenados, y=[str(s.date()) for s in safras_ordenadas],
                       colorscale="Reds", zmin=0, zmax=painel["taxa_entrada"].quantile(0.98)),
            go.Scatter(x=sub["mes_calendario"], y=sub["c_mais"], mode="lines+markers",
                      line=dict(color="#B23A48", width=2)),
            go.Scatter(x=sub["mes_calendario"], y=sub["c_menos"], mode="lines+markers",
                      line=dict(color="#2A6F97", width=2)),
        ],
    ))
fig.frames = frames

steps = [
    dict(method="animate", label=str(mes_foto.date()),
         args=[[str(mes_foto.date())], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                             transition=dict(duration=0))])
    for mes_foto in meses_unicos
]

fig.update_layout(
    width=1400, height=650,
    title="Evolucao fotografia por fotografia (arraste o controle abaixo)",
    xaxis_title="MOB (idade do contrato)", yaxis_title="Safra (mes de originacao)",
    xaxis2_title="Fotografia (mes de referencia)", yaxis2_title="Estatistica de Page",
    template="plotly_white",
    sliders=[dict(active=0, currentvalue=dict(prefix="Fotografia: "), steps=steps)],
    updatemenus=[dict(type="buttons", showactive=False, y=1.15, x=1.05,
                       buttons=[dict(label="Reproduzir", method="animate",
                                     args=[None, dict(frame=dict(duration=350, redraw=True),
                                                       fromcurrent=True)])])],
)
fig.show()



## 8. Efeito de safra (com alerta de credibilidade)

Ajustado com todo o histórico. `veio_truncado` **não** entra aqui
(colinear com `C(safra)` para as safras pré-janela). O ranking exibido
fica restrito às safras dentro da janela -- são as únicas em que
"qualidade de subscrição" é acionável.


In [13]:

FORMULA_MODELO_B = "entrantes_90mais ~ C(mob_bin) + macro_reportado + C(safra)" if USA_MACRO \
    else "entrantes_90mais ~ C(mob_bin) + C(safra)"

modelo_b = smf.glm(
    formula=FORMULA_MODELO_B, data=painel,
    family=sm.families.Poisson(), offset=np.log(painel["populacao_risco"]),
).fit()

safras_janela = sorted(painel.loc[painel["veio_truncado"] == 0, "safra"].unique())
meses_observados_por_safra = painel.groupby("safra")["mob"].count()
efeitos = []
for s in safras_janela:
    nome_param = f"C(safra)[T.{repr(s)}]"  # patsy nomeia categorias de data como Timestamp('...'), nao a data pura
    if nome_param in modelo_b.params.index:
        coef, erro_padrao = modelo_b.params[nome_param], modelo_b.bse[nome_param]
    else:
        coef, erro_padrao = 0.0, 0.0
    efeitos.append(dict(safra=s, efeito=coef, erro_padrao=erro_padrao,
                         meses_observados=int(meses_observados_por_safra.get(s, 0))))

efeitos_df = pd.DataFrame(efeitos)
efeitos_df["credivel"] = efeitos_df["meses_observados"] >= LIMIAR_CREDIVEL
cores = np.where(efeitos_df["credivel"], "#B23A48", "#D9B8BC")

fig_safra = go.Figure()
fig_safra.add_trace(go.Bar(
    x=[str(s.date()) for s in efeitos_df["safra"]], y=efeitos_df["efeito"],
    error_y=dict(type="data", array=1.96 * efeitos_df["erro_padrao"], visible=True),
    marker_color=cores,
    text=[f"{m} meses" for m in efeitos_df["meses_observados"]],
    hovertemplate="Safra %{x}<br>Efeito: %{y:.3f}<br>%{text}<extra></extra>",
))
fig_safra.add_hline(y=0, line_dash="dot", line_color="gray")
fig_safra.update_layout(
    title=f"Efeito de safra (dentro da janela) -- tom claro = menos de {LIMIAR_CREDIVEL} meses observados",
    xaxis_title="Safra (mes de originacao)", yaxis_title="Efeito estimado (log-odds relativo)",
    template="plotly_white", width=1400, height=450,
)
fig_safra.show()

print("Safras com maior efeito estimado (entre as confiaveis):")
print(efeitos_df[efeitos_df["credivel"]].sort_values("efeito", ascending=False).head(6)
      [["safra", "efeito", "meses_observados"]])


Safras com maior efeito estimado (entre as confiaveis):
        safra    efeito  meses_observados
18 2023-07-01  1.290665                 6
10 2022-11-01  1.202240                14
9  2022-10-01  1.127588                15
17 2023-06-01  1.085579                 7
15 2023-04-01  1.081601                 9
8  2022-09-01  1.067256                16



## Notas finais

**Sobre `id_cliente`**: não entra em nada deste notebook. Ele importaria
para uma pergunta diferente -- se o default de um produto de um cliente
eleva o risco de outro produto do mesmo cliente (propagação dentro do
mesmo CPF) -- que foi levantada e deixada de lado nesta conversa por falta
de dado suficiente de sobreposição de produtos para testar com poder
estatístico. Se isso voltar a ser relevante, é uma extensão separada, não
algo a forçar dentro deste diagnóstico de safra/idade/época.

**Limites que continuam valendo:**
- Calibração do `h` (Seção 7) usa um processo nulo simplificado (escala
  ajustada ao seu painel, mas sem estoque herdado nem cura/reentrada).
- `C-` cruzando `h` não é prova de melhora real -- pode ser o modelo se
  recalibrando ao patamar já ruim.
- Sem `df_macro` real, `USA_MACRO=False` é a opção honesta -- não
  substitua por uma variável fabricada.
- Elegibilidade (Seção 2) descarta contratos truncados com `mob>1` e sem
  mês anterior conhecido -- population at risk nessas células fica menor
  do que o estoque bruto, por desenho, não por erro.
